#Quick to Draw? More Like Quick to Classify!

Data set: https://github.com/googlecreativelab/quickdraw-dataset/blob/master/README.md


Problem Statement:
We are using this dataset to explore different classification models as well within those models different activation functions and other variables in how they function to see how they affect performance.

Methodologies:
Through the implementation of different models (So far the only one we know we want to work with is CNN) we will be comparing and contrasting their performance through both how long it takes to train and run the model on the data as well as, how accurately it classifies the images. We are going to take a subset of the data both in terms of raw image count as well as categories of images and using this “new dataset” we will train our models just to make our lives easier.

GitHub:
https://github.com/Mar3759/CSCI335_Group_4_Final_Project

---

##Data Preprocessing
You should carefully explore the dataset and apply appropriate preprocessing steps. Perform data cleaning, including handling missing values, addressing class imbalance, applying normalization or encoding techniques, and conducting feature engineering or selection, train/test split as needed.

### 1. Get the Data

We will be using the following classes as a subset of the full dataset:
`Horse, Zebra, Shoe, Rollerskate, Cat, Dog, Panda,
Laptop, Computer, Camera, Television, Mushroom, Strawberry, Bread, Bandage, Baseball, Bat, Crab, Clock, Compass, Fan, Lollipop, Soccer ball, Basketball, Stop Sign, Octagon`


We are using the "simplified drawing files" data provided, in which the images are saved in simplified vector strokes where the timing information is removed, and the data has been scaled and positioned into a 256x256 region. The data is in ndjson format with the same metadata as the raw format.

The simplification process was:
- Align the drawing to the top-left corner, to have minimum values of 0.
- Uniformly scale the drawing, to have a maximum value of 255.
- Resample all strokes with a 1 pixel spacing.
- Simplify all strokes using the Ramer–Douglas–Peucker algorithm with an epsilon value of 2.0.

In [ ]:
classes = [
  "horse", "zebra", "shoe", "rollerskates", "cat", "dog", "panda",
  "laptop", "computer", "camera", "television",
  "mushroom", "strawberry", "bread", "bandage",
  "baseball", "bat", "crab", "clock", "compass", "fan",
  "lollipop", "soccer ball", "basketball",
  "stop sign", "octagon"
]

!mkdir quickdraw

for c in classes:
  !gsutil cp "gs://quickdraw_dataset/full/simplified/{c}.ndjson" quickdraw/

mkdir: cannot create directory ‘quickdraw’: File exists
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://quickdraw_dataset/full/simplified/horse.ndjson...
- [1 files][ 99.8 MiB/ 99.8 MiB]                                                
Operation completed over 1 objects/99.8 MiB.                                     
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://quickdraw_dataset/full/simplified/zebra.ndjson...
\
Operation completed over 1 objects/92.1 MiB.                                     
Google recommends using Gcloud storage CLI (https://d

Data is saved in vector strokes:

In [ ]:
import pandas as pd

df = pd.read_json("quickdraw/horse.ndjson", lines=True)
df.head()

,word,countrycode,timestamp,recognized,key_id,drawing
0,horse,RU,2017-01-07 12:20:35.937330+00:00,True,5717083633483776,"[[[49, 70, 85, 92, 110, 130, 173, 218, 235, 24..."
1,horse,US,2017-03-11 23:28:01.969980+00:00,False,5757266059853824,"[[[11, 2, 3, 14, 32, 80, 74, 70, 116, 131, 140..."
2,horse,US,2017-03-03 00:02:16.303920+00:00,True,5869593245515776,"[[[98, 93, 93], [145, 104, 82]], [[101, 91, 90..."
3,horse,US,2017-03-18 20:37:39.344110+00:00,True,6699365039079424,"[[[31, 16, 4, 0, 3, 23, 59, 67, 75, 73, 49, 66..."
4,horse,US,2017-03-19 18:52:56.580340+00:00,True,5247220423065600,"[[[67, 64, 66, 71, 76, 81, 85, 88, 90], [89, 1..."


### 2. Convert the Data as vector strokes to raster image in pixels to binary array

In [ ]:
import numpy as np
import cv2

def vector_drawing_to_binary(drawing_vector):

  #initialize a 256 x 256 blank image with pixels of 0
  image = np.zeros((256, 256), dtype=np.uint8)

  for stroke in drawing_vector:

    #get the x and y coords of each stroke
    x = stroke[0]
    y = stroke[1]

    for i in range(len(x)-1):

      #get each pair of points in the stroke
      point1 = (x[i], y[i])
      point2 = (x[i+1], y[i+1])

      #draw a line between the two points on the image with open cv
      image = cv2.line(image, point1, point2, 255, 8)

  #create a boolean array where true for pixels > 0 and false for 0 or less
  drawing_boolean_array = (image > 0)

  #convert the boolean values into bit unsigned integer format
  drawing_binary = drawing_boolean_array.astype(np.uint8)

  #resize image to 28x28 instead of 256x256
  resized_drawing_binary = cv2.resize(drawing_binary, (28, 28))

  return resized_drawing_binary

Test with First Image from each Class

In [ ]:
import json

classes = [
  "horse", "zebra", "shoe", "rollerskates", "cat", "dog", "panda",
  "laptop", "computer", "camera", "television",
  "mushroom", "strawberry", "bread", "bandage",
  "baseball", "bat", "crab", "clock", "compass", "fan",
  "lollipop", "soccer ball", "basketball",
  "stop sign", "octagon"
]

#binary images
X = []
#labels
y = []

for c in classes:
  filepath = f"quickdraw/{c}.ndjson"
  print("Processing data in:", filepath)
  with open(filepath) as f:
    #process each drawing in the class
    for line in f:
      data = json.loads(line)
      drawing = data["drawing"]
      binary_drawing = vector_drawing_to_binary(drawing)

      #save binary drawing to list
      X.append(binary_drawing)
      #save label to list
      y.append(c)
      break

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Labels shape:", y.shape)

Processing data in: quickdraw/horse.ndjson
Processing data in: quickdraw/zebra.ndjson
Processing data in: quickdraw/shoe.ndjson
Processing data in: quickdraw/rollerskates.ndjson
Processing data in: quickdraw/cat.ndjson
Processing data in: quickdraw/dog.ndjson
Processing data in: quickdraw/panda.ndjson
Processing data in: quickdraw/laptop.ndjson
Processing data in: quickdraw/computer.ndjson
Processing data in: quickdraw/camera.ndjson
Processing data in: quickdraw/television.ndjson
Processing data in: quickdraw/mushroom.ndjson
Processing data in: quickdraw/strawberry.ndjson
Processing data in: quickdraw/bread.ndjson
Processing data in: quickdraw/bandage.ndjson
Processing data in: quickdraw/baseball.ndjson
Processing data in: quickdraw/bat.ndjson
Processing data in: quickdraw/crab.ndjson
Processing data in: quickdraw/clock.ndjson
Processing data in: quickdraw/compass.ndjson
Processing data in: quickdraw/fan.ndjson
Processing data in: quickdraw/lollipop.ndjson
Processing data in: quickdraw

Print Frist Image from each Class

In [ ]:
for i in range(len(X)):
  print(y[i])
  print(X[i])

horse
[[0 0 0 0 0 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 1 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 0 0 0 0 1 1 1 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0]
 [1 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0]
 [0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1]
 [0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1]
 [0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 1 1]
 [0 0 0 0 0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 1 0 0 0 1 1]
 [0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 1 0 

### 3. Process all the Data from each Class

In [ ]:
classes = [
  "horse", "zebra", "shoe", "rollerskates", "cat", "dog", "panda",
  "laptop", "computer", "camera", "television",
  "mushroom", "strawberry", "bread", "bandage",
  "baseball", "bat", "crab", "clock", "compass", "fan",
  "lollipop", "soccer ball", "basketball",
  "stop sign", "octagon"
]

#binary images
X = []
#labels
y = []

for c in classes:
  filepath = f"quickdraw/{c}.ndjson"
  print("Processing data in:", filepath)
  with open(filepath) as f:
    #process each data point / drawing in the class
    for line in f:
      data = json.loads(line)
      drawing = data["drawing"]
      binary_drawing = vector_drawing_to_binary(drawing)

      #save binary drawing to list
      X.append(binary_drawing)
      #save label to list
      y.append(c)

X = np.array(X)
y = np.array(y)

print("Dataset shape:", X.shape)
print("Labels shape:", y.shape)

Processing data in: quickdraw/horse.ndjson
Processing data in: quickdraw/zebra.ndjson
Processing data in: quickdraw/shoe.ndjson
Processing data in: quickdraw/rollerskates.ndjson
Processing data in: quickdraw/cat.ndjson
Processing data in: quickdraw/dog.ndjson
Processing data in: quickdraw/panda.ndjson
Processing data in: quickdraw/laptop.ndjson
Processing data in: quickdraw/computer.ndjson
Processing data in: quickdraw/camera.ndjson
Processing data in: quickdraw/television.ndjson
Processing data in: quickdraw/mushroom.ndjson
Processing data in: quickdraw/strawberry.ndjson
Processing data in: quickdraw/bread.ndjson
Processing data in: quickdraw/bandage.ndjson
Processing data in: quickdraw/baseball.ndjson
Processing data in: quickdraw/bat.ndjson
Processing data in: quickdraw/crab.ndjson
Processing data in: quickdraw/clock.ndjson
Processing data in: quickdraw/compass.ndjson
Processing data in: quickdraw/fan.ndjson
Processing data in: quickdraw/lollipop.ndjson
Processing data in: quickdraw

Number of Data points in each Class:

In [ ]:
from collections import Counter
Counter(y)

Counter({np.str_('horse'): 178286,
         np.str_('zebra'): 144608,
         np.str_('shoe'): 120231,
         np.str_('rollerskates'): 119772,
         np.str_('cat'): 123202,
         np.str_('dog'): 152159,
         np.str_('panda'): 113613,
         np.str_('laptop'): 261501,
         np.str_('computer'): 123885,
         np.str_('camera'): 128772,
         np.str_('television'): 123137,
         np.str_('mushroom'): 142167,
         np.str_('strawberry'): 122301,
         np.str_('bread'): 120570,
         np.str_('bandage'): 147614,
         np.str_('baseball'): 135375,
         np.str_('bat'): 118114,
         np.str_('crab'): 126930,
         np.str_('clock'): 120536,
         np.str_('compass'): 127609,
         np.str_('fan'): 136158,
         np.str_('lollipop'): 128849,
         np.str_('soccer ball'): 125349,
         np.str_('basketball'): 133793,
         np.str_('stop sign'): 119814,
         np.str_('octagon'): 159474})

### Next To Do:
- Check if any drawings are blank and remove
- Encode the class labels into numbers
- See if any of the classes are imbalanced and have more/less data (Computer looks like it has a lot more)
- Split the data into training/testing sets


---

##Modeling
Train and evaluate at least three distinct machine learning models, including both traditional and neural network-based models. If applicable, perform hyperparameter tuning to optimize model performance, model selection, cross-validation, and ensemble models


# 1. First attempt at basic Multi-layer Perceptron Model



In [ ]:
from sklearn.model_selection import train_test_split

mlp_X = X
mlp_y = pd.get_dummies(y, drop_first=False)

mlp_X_train, mlp_X_test, mlp_y_train, mlp_y_test = train_test_split(mlp_X, mlp_y,test_size = 0.1,random_state=1)




---

##Evaluation

Use appropriate evaluation metrics to examine training and test performance, detecting underfitting or overfitting, and compare the strengths and weaknesses of the models.


---

##Result Analysis
Provide a critical analysis of the approaches used, justify which method performed best for the given dataset, and discuss the limitations of your work as well as potential avenues for improvement.